# Imports

In [6]:
from idf_analysis.analysis.historical.bartlett_lewis import BartlettLewisModel, disaggregate_daily_to_subdaily
from idf_analysis.data.processing import read_csv, aggregate_to_csv, save_to_csv, verification, fill_missing_data

from idf_analysis.helpers.notebook import precip_summary
import pandas as pd

# Criação de precipitações sintéticas com Bartlett-Lewis

## Preparação de dados para a calibração

Obs.: Quanto mais fino o intervalo de dados disponível, melhor

In [5]:
cemaden_df = read_csv(path='../results/cemaden_ac_santana_sao/cemaden_ac_santana_sao_hourly.csv')
inmet_df = read_csv(path='../results/inmet_sao_paulo_mirante/inmet_sao_paulo_mirante_hourly.csv')

summary_hourly = pd.concat([
    precip_summary(inmet_df, "INMET Horário"),
    precip_summary(cemaden_df, "CEMADEN"),
    
])
print("📊 Base de dados para IDF's históricas: ")
display(summary_hourly.style.format(precision=2))

📊 Base de dados para IDF's históricas: 


,Total registros,Registros > 0,Primeira data,Última data,Precipitação média (mm),Precipitação máxima (mm)
INMET Horário,96432,7266,2014-01-01 00:00:00,2024-12-31 23:00:00,0.17,77.80
CEMADEN,61305,6038,2014-01-01 00:00:00,2024-12-31 23:00:00,0.12,45.00


In [7]:
df_hourly = fill_missing_data(
    df_main=cemaden_df,
	df_secondary=inmet_df,
    frequency='hourly'
)

# Verificando novamente após o preenchimento
verification(df=df_hourly, frequency='hourly');

# Conjuga as colunas Year, Month, Day em uma coluna timestamp
df_hourly['timestamp'] = pd.to_datetime(df_hourly[['Year', 'Month', 'Day', 'Hour']])
df_hourly.set_index('timestamp', inplace=True)

display(df_hourly.head())

preprocessed_file = 'cemaden_ac_santana_sao_preprocessed'
preprocessed_path = '../results/cemaden_ac_santana_sao/'
df_hourly.to_csv(f'{preprocessed_path}{preprocessed_file}_hourly.csv', index=True)

[OK] Série completa! Nenhum período faltando.



,Year,Month,Day,Hour,Precipitation,Date
timestamp,,,,,,
2014-01-01 00:00:00,2014,1,1,0,0.0,2014-01-01 00:00:00
2014-01-01 01:00:00,2014,1,1,1,0.0,2014-01-01 01:00:00
2014-01-01 02:00:00,2014,1,1,2,0.0,2014-01-01 02:00:00
2014-01-01 03:00:00,2014,1,1,3,0.0,2014-01-01 03:00:00
2014-01-01 04:00:00,2014,1,1,4,0.0,2014-01-01 04:00:00


In [8]:
# Inicializar modelo
bl_model = BartlettLewisModel()

In [9]:
df_hourly.head(30)

,Year,Month,Day,Hour,Precipitation,Date
timestamp,,,,,,
2014-01-01 00:00:00,2014,1,1,0,0.00,2014-01-01 00:00:00
2014-01-01 01:00:00,2014,1,1,1,0.00,2014-01-01 01:00:00
2014-01-01 02:00:00,2014,1,1,2,0.00,2014-01-01 02:00:00
2014-01-01 03:00:00,2014,1,1,3,0.00,2014-01-01 03:00:00
2014-01-01 04:00:00,2014,1,1,4,0.00,2014-01-01 04:00:00
2014-01-01 05:00:00,2014,1,1,5,0.00,2014-01-01 05:00:00
2014-01-01 06:00:00,2014,1,1,6,0.00,2014-01-01 06:00:00
2014-01-01 07:00:00,2014,1,1,7,0.00,2014-01-01 07:00:00
2014-01-01 08:00:00,2014,1,1,8,0.00,2014-01-01 08:00:00


## Inicialização do modelo e identificação de eventos

In [10]:
# Identificar eventos
print("\n🔍 Identificando eventos de precipitação...")
events = bl_model.identify_events(
	df_hourly['Precipitation'],
	inter_event_gap_minutes=60
)

print(f"   ✓ {len(events)} eventos identificados")


🔍 Identificando eventos de precipitação...
   ✓ 4982 eventos identificados


## Calibração do modelo

In [25]:
print("\n⚙️  Calibrando modelo...")
params = bl_model.calibrate(
    events=events,
    interval_minutes=60,
    intra_event_gap_minutes=60,
)

print("\n📊 Parâmetros Calibrados:")
for key, value in params.items():
	print(f"   {key:8s}: {value:.4f}")

# Exportar parâmetros
output_file = '../results/cemaden_ac_santana_sao/bartlett_lewis_params.yaml'
bl_model.export_params(output_file)
print(f"\n✓ Parâmetros salvos em: {output_file}")


⚙️  Calibrando modelo...

📊 Parâmetros Calibrados:
   lambda  : 1.2405
   beta    : 1.0000
   gamma   : 0.0031
   eta     : 0.0031
   mu      : 0.0079
✅ Parâmetros exportados para: ../results/cemaden_ac_santana_sao/bartlett_lewis_params.yaml

✓ Parâmetros salvos em: ../results/cemaden_ac_santana_sao/bartlett_lewis_params.yaml


## Geração da chuva sintética com o modelo

In [12]:
print("\n🌧️  Gerando chuva sintética...")
synthetic_rain = bl_model.generate_synthetic_rainfall(
	total_duration_minutes=1440*365*11,  # 11 anos de chuva
	output_interval_minutes=10,
	initial_timestamp=df_hourly.index.min(), # Iniciando a chuva sintética na data inicial dos dados reais
	seed=None
)
synthetic_rain.to_csv('../results/bl_chuva_sintetica.csv')
print("Synthetic rainfall generated and saved.")


🌧️  Gerando chuva sintética...
Synthetic rainfall generated and saved.


# Agregação da chuva sintética

### Formatação dos dados para utilização da função de agregação

In [13]:
# Separa a coluna timestamp em Year, Month, Day, Hour e Minute
st_rain_formatted = synthetic_rain.reset_index()
st_rain_formatted.columns = ['timestamp', 'rainfall_mm']
st_rain_formatted['Year'] = st_rain_formatted['timestamp'].dt.year
st_rain_formatted['Month'] = st_rain_formatted['timestamp'].dt.month
st_rain_formatted['Day'] = st_rain_formatted['timestamp'].dt.day
st_rain_formatted['Hour'] = st_rain_formatted['timestamp'].dt.hour
st_rain_formatted['Minute'] = st_rain_formatted['timestamp'].dt.minute

# Renomeia a coluna rainfall_mm para Precipitation
st_rain_formatted = st_rain_formatted[['Year', 'Month', 'Day', 'Hour', 'Minute', 'rainfall_mm']]
st_rain_formatted.columns = ['Year', 'Month', 'Day', 'Hour', 'Minute', 'Precipitation']


In [14]:
st_rain_formatted.head()

,Year,Month,Day,Hour,Minute,Precipitation
0,2014,1,1,0,0,0.0
1,2014,1,1,0,10,0.0
2,2014,1,1,0,20,0.0
3,2014,1,1,0,30,0.0
4,2014,1,1,0,40,0.0


In [15]:
bl_rain_name = 'cemaden_ac_santana_sao_synthetic'
output_path = '../results/cemaden_ac_santana_sao/'

aggregate_to_csv(
	st_rain_formatted,
	name=bl_rain_name,
	directory=output_path,
	include_minutes=True,
	minute_freq=60
)
print(f"\n✓ Chuva sintética salva em: {output_path}{bl_rain_name}_hourly.csv")


✓ Chuva sintética salva em: ../results/cemaden_ac_santana_sao/cemaden_ac_santana_sao_synthetic_hourly.csv


In [16]:
df_synthetic_hourly = read_csv(f"{output_path}{bl_rain_name}_hourly.csv")
df_synthetic_hourly.head()

,Year,Month,Day,Hour,Precipitation
0,2014,1,1,0,0.0
1,2014,1,1,1,0.0
2,2014,1,1,2,0.0
3,2014,1,1,3,0.0
4,2014,1,1,4,0.0


# Comparação da chuva sintética com a chuva original

In [17]:
# Escreve o tamanho do dataframe
print(f"\n✓ Dados horários sintéticos carregados: {len(df_synthetic_hourly)} registros")


✓ Dados horários sintéticos carregados: 96360 registros


In [18]:
from idf_analysis.analysis.historical.validation import max_annual_precipitation

max_annual_cemaden = max_annual_precipitation(df=df_hourly, name_file='cemaden_ac_santana_sao', output_dir='../results/cemaden_bl_comparison/', frequency='hourly')
max_annual_synthetic = max_annual_precipitation(df=df_synthetic_hourly, name_file='cemaden_ac_santana_sao_bartlett_lewis_synthetic', output_dir='../results/cemaden_bl_comparison/', frequency='hourly')

print("\n📊 CEMADEN - Máximos anuais:")
display(max_annual_cemaden)
print("\n📊 Chuva Sintética - Máximos anuais:")
display(max_annual_synthetic)


📊 CEMADEN - Máximos anuais:


,Year,Precipitation
0,2014,67.3700
1,2015,104.3300
2,2016,58.8697
3,2017,84.6900
4,2018,70.1800
5,2019,35.9700
6,2020,49.7955
7,2021,40.1047
8,2022,58.2400
9,2023,55.2333



📊 Chuva Sintética - Máximos anuais:


,Year,Precipitation
0,2014,4.058290
1,2015,3.358935
2,2016,3.941444
3,2017,4.653168
4,2018,5.531609
5,2019,3.645492
6,2020,6.672795
7,2021,5.120172
8,2022,2.843728
9,2023,3.833398


In [19]:
from idf_analysis.analysis.historical.subdaily import get_max_subdaily_table

incomplete_subdaily_cemaden = get_max_subdaily_table(df=df_hourly, name_file='cemaden_ac_santana_sao', output_dir='../results/cemaden_bl_comparison/')

print("\n📊 CEMADEN - Máximos subdiários:")
display(incomplete_subdaily_cemaden)

incomplete_subdaily_synthetic = get_max_subdaily_table(df=df_synthetic_hourly, name_file='synthetic_rainfall', output_dir='../results/cemaden_bl_comparison/')

print("\n📊 Bartlett-Lewis - Máximos subdiários:")
display(incomplete_subdaily_synthetic)


[OK] Resultados salvos em: ../results/cemaden_bl_comparison//max_subdaily_cemaden_ac_santana_sao.csv

📊 CEMADEN - Máximos subdiários:


,Year,Max_1h,Max_3h,Max_6h,Max_8h,Max_10h,Max_12h,Max_24h
0,2014,45.00,55.46,63.04,64.22,67.36,68.36,119.54
1,2015,43.99,89.54,94.08,94.28,94.48,94.48,104.33
2,2016,25.82,40.83,52.29,59.26,61.19,61.34,65.00
3,2017,26.84,55.56,67.17,67.97,68.35,70.52,87.44
4,2018,12.20,17.70,26.50,34.36,42.61,49.68,74.89
5,2019,25.98,31.88,34.25,36.14,36.14,36.34,46.46
6,2020,12.40,17.54,32.20,32.85,33.09,40.44,50.97
7,2021,11.14,21.41,29.53,37.09,42.20,44.06,57.36
8,2022,37.38,44.45,44.65,44.65,44.65,44.65,62.58
9,2023,30.74,41.03,43.01,43.81,43.81,44.20,56.44



[OK] Resultados salvos em: ../results/cemaden_bl_comparison//max_subdaily_synthetic_rainfall.csv

📊 Bartlett-Lewis - Máximos subdiários:


,Year,Max_1h,Max_3h,Max_6h,Max_8h,Max_10h,Max_12h,Max_24h
0,2014,1.43,2.76,3.73,4.06,4.06,4.06,4.06
1,2015,1.64,2.80,3.16,3.16,3.17,3.17,3.67
2,2016,1.79,3.60,4.37,4.37,4.61,4.68,4.68
3,2017,1.41,2.97,3.62,3.81,3.81,3.99,4.65
4,2018,1.52,3.10,5.28,5.53,5.53,5.53,7.47
5,2019,1.47,3.02,3.51,3.51,3.51,3.52,3.65
6,2020,1.34,3.51,6.24,6.36,6.45,6.57,8.22
7,2021,2.33,4.66,5.12,5.12,5.12,5.12,5.17
8,2022,1.71,2.55,2.84,3.09,3.09,3.09,3.54
9,2023,1.72,3.68,3.68,3.68,3.68,3.68,3.83
